# 니나브봇 Gemma 4 E4B 무료 Colab QLoRA

무료 Colab T4 16GB에 맞춘 설정입니다. 학습 결과와 checkpoint는 Google Drive에 저장합니다.

실행 전 `런타임 > 런타임 유형 변경 > T4 GPU`를 선택하세요. 무료 Colab GPU는 보장되지 않으며 사용 중 런타임이 종료될 수 있습니다. 종료되면 같은 셀을 다시 실행해 마지막 checkpoint부터 이어갑니다.

이 노트북은 니나브봇의 도구 라우터를 학습합니다. 니나브 캐릭터 대화용 데이터는 별도 학습으로 분리하는 편이 안전합니다.

In [ ]:
# GPU가 아니면 8B급 QLoRA가 사실상 진행되지 않으므로 시작 전에 막습니다.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch

if not torch.cuda.is_available():
    raise RuntimeError("런타임 유형에서 T4 GPU를 선택하세요.")

props = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")

In [ ]:
# Colab 기본 패키지 조합은 자주 바뀌므로 한 셀에서 호환 버전을 다시 맞춥니다.
%pip install -q -U unsloth unsloth_zoo datasets trl peft accelerate

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/ninav-gemma4")
DATA_PATH = DRIVE_ROOT / "train.json"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
FINAL_ADAPTER_DIR = DRIVE_ROOT / "adapter_final"

MODEL_SIZE = "E4B"
MODEL_NAME = f"unsloth/gemma-4-{MODEL_SIZE}-it-unsloth-bnb-4bit"
MAX_LENGTH = 512
LORA_RANK = 8
EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 5e-5

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"작업 폴더: {DRIVE_ROOT}")
print(f"모델: {MODEL_NAME}")
print("E4B가 계속 OOM이면 MODEL_SIZE를 E2B로 바꾸세요.")

## 학습 데이터 업로드

처음 실행할 때 `router_seed.json` 또는 보강한 `train.json`을 선택하세요. 업로드한 파일은 Drive의 `ninav-gemma4/train.json`으로 복사되므로 다음 실행부터 다시 올릴 필요가 없습니다.

In [ ]:
import json
from google.colab import files

if not DATA_PATH.exists():
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("학습 JSON 파일을 선택하세요.")
    name, content = next(iter(uploaded.items()))
    DATA_PATH.write_bytes(content)
    print(f"{name} -> {DATA_PATH}")

raw_data = json.loads(DATA_PATH.read_text(encoding="utf-8"))
if not isinstance(raw_data, list) or not raw_data:
    raise ValueError("학습 데이터는 비어 있지 않은 JSON 배열이어야 합니다.")

for index, row in enumerate(raw_data):
    messages = row.get("messages") if isinstance(row, dict) else None
    if not isinstance(messages, list) or len(messages) < 2:
        raise ValueError(f"{index}번 데이터에 messages가 없습니다.")
    assistant = messages[-1]
    if assistant.get("role") != "assistant":
        raise ValueError(f"{index}번 데이터의 마지막 메시지가 assistant가 아닙니다.")
    try:
        answer = json.loads(assistant.get("content", ""))
    except json.JSONDecodeError as exc:
        raise ValueError(f"{index}번 assistant 응답이 JSON이 아닙니다.") from exc
    if not isinstance(answer, dict) or not ({"tool_calls", "text"} & answer.keys()):
        raise ValueError(f"{index}번 응답에 tool_calls 또는 text가 없습니다.")

print(f"데이터: {len(raw_data)}개")
if len(raw_data) < 100:
    print("주의: 현재 데이터는 파이프라인 시험용입니다. 실사용 전 수백 개로 보강하세요.")

In [ ]:
import gc
from unsloth import FastModel

gc.collect()
torch.cuda.empty_cache()

# T4에서는 Gemma 4 임베딩이 FP32로 승격되므로 CPU에 두지 않으면 로드 중 OOM이 납니다.
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
    offload_embedding=True,
    text_only=True,
)

model = FastModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"학습 파라미터: {trainable / 1e6:.1f}M")
print(f"현재 VRAM: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

ROUTER_SYSTEM = (
    "너는 로스트아크 디스코드 봇의 라우터다. 한국어 요청에서 실행할 도구와 인자를 고른다. "
    "반드시 JSON 하나만 출력한다. 도구를 실행할 수 있으면 "
    '{"tool_calls":[{"name":"도구명","arguments":{}}]}, 정보가 부족하거나 지원하지 않으면 '
    '{"text":"한국어 안내"} 형식을 쓴다. 이모지를 쓰지 않는다.'
)

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

def format_example(example):
    messages = [{"role": "system", "content": ROUTER_SYSTEM}, *example["messages"]]
    return {
        "text": tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    }

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_example, remove_columns=dataset.column_names)
print(dataset[0]["text"][:500])

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir=str(CHECKPOINT_DIR),
        dataset_text_field="text",
        max_length=MAX_LENGTH,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        logging_steps=1,
        save_strategy="steps",
        save_steps=25,
        save_total_limit=2,
        report_to="none",
        seed=42,
    ),
)
print(f"유효 배치: {BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
# Drive checkpoint가 있으면 무료 런타임 종료 전 지점에서 이어갑니다.
def checkpoint_step(path):
    try:
        return int(path.name.rsplit("-", 1)[-1])
    except ValueError:
        return -1

checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"), key=checkpoint_step)
resume = str(checkpoints[-1]) if checkpoints else None
print(f"재개 지점: {resume or '처음부터'}")

stats = trainer.train(resume_from_checkpoint=resume)
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"완료: loss={stats.metrics['train_loss']:.4f}")
print(f"adapter: {FINAL_ADAPTER_DIR}")

## 결과 확인

학습 데이터에 없던 문장으로 JSON 형식과 도구 선택을 확인합니다. 이 셀 결과가 계속 깨지면 데이터를 늘린 뒤 다시 학습하세요.

In [ ]:
model.eval()

def ask_router(question):
    messages = [
        {"role": "system", "content": ROUTER_SYSTEM},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text=[prompt], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    return tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

for question in [
    "카제로스 서버에 웨이 카드 뜨면 알려줘",
    "8명이서 25만 골드에 먹었어",
    "재련 평균 비용 계산해줘",
]:
    print(f"Q: {question}")
    print(f"A: {ask_router(question)}\n")

In [ ]:
# Drive에도 남지만 로컬 PC로 바로 받을 수 있게 작은 adapter만 압축합니다.
import shutil
from google.colab import files

archive = shutil.make_archive("/content/ninav-router-adapter", "zip", FINAL_ADAPTER_DIR)
print(archive)
files.download(archive)